
# 🏠 Kaggle Starter: House Prices — Advanced Regression Techniques
**Autor:** Eugenia Rusu  
**Obiectiv:** Baseline solid pentru regresie + fișier `submission.csv` gata de urcat.  
**Link competiție:** https://www.kaggle.com/c/house-prices-advanced-regression-techniques

## Ce conține notebook-ul
- EDA de bază (lipsuri, distribuții, țintă log-transform)  
- Feature engineering minim (imputări, categorice)  
- Modelare cu `ElasticNet` și `XGBoost` (opțional)  
- Cross-validation cu `KFold` și metrică RMSE (scor oficial Kaggle)  
- Export `submission.csv`



## 1. Setup & Pachete


In [ ]:

# !pip install pandas numpy scikit-learn matplotlib xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

pd.set_option("display.max_columns", None)
print("Pachete încărcate ✅", " (XGBoost OK)" if HAS_XGB else " (XGBoost indisponibil)")



## 2. Încărcarea datelor
Pe Kaggle, datele sunt în `/kaggle/input/house-prices-advanced-regression-techniques/`.


In [ ]:

# CĂI IMPLICITE (Kaggle)
TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

# Dacă rulezi local, setează manual:
# TRAIN_PATH = "train.csv"
# TEST_PATH  = "test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(train.shape, test.shape)
train.head()



## 3. EDA rapid
- Ținta este `SalePrice` (pozitivă, skewed).  
- Pentru RMSE stabil, folosim log-transform pe țintă la antrenare/validare.


In [ ]:

# Lipsuri
missing = train.isna().mean().sort_values(ascending=False)
display(missing.head(15).to_frame("proportion_missing"))

# Distribuția țintei
train["SalePrice"].hist(bins=30)
plt.title("Distribuția SalePrice (train)")
plt.xlabel("SalePrice"); plt.ylabel("Count")
plt.show()

print(train["SalePrice"].describe())



## 4. Feature Engineering minim
- Imputări simple pentru numeric și categoric.  
- One-hot encoding pentru categorice.  
- Model de bază: `ElasticNet` (bun pentru regresie tabulară, interpretabil).


In [ ]:

TARGET = "SalePrice"
ID_COL = "Id"

# Separăm ținta și aplicăm log-transform (stabilizează RMSE)
y = np.log1p(train[TARGET])
train_features = train.drop(columns=[TARGET])

# Separăm tipurile de coloane
numeric_cols = train_features.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_features.select_dtypes(exclude=[np.number]).columns.tolist()

X = train_features[numeric_cols + categorical_cols].copy()
X_test = test[numeric_cols + categorical_cols].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Model bază: ElasticNet (mix L1/L2)
model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)

pipe = Pipeline(steps=[("preprocess", preprocess),
                      ("model", model)])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(pipe, X, y):
    # Scoring custom: transformăm din MSE -> RMSE
    neg_mse = cross_val_score(pipe, X, y, scoring="neg_mean_squared_error", cv=kf)
    rmse = np.sqrt(-neg_mse)
    return rmse

rmse_scores = rmse_cv(pipe, X, y)
print("ElasticNet CV RMSE (log-target):", rmse_scores.mean().round(4), "±", rmse_scores.std().round(4))



## 5. Antrenare finală & `submission.csv`
Predicțiile sunt în spațiul log — transformăm înapoi cu `expm1`.


In [ ]:

pipe.fit(X, y)
preds_log = pipe.predict(X_test)
preds = np.expm1(preds_log)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: preds
})
submission_path = "submission_house_prices_elasticnet.csv"
submission.to_csv(submission_path, index=False)
print(f"Am salvat: {submission_path}")
submission.head()



## 6. (Opțional) XGBoost pentru performanță mai bună
Dacă `xgboost` este instalat, testăm rapid un model non-liniar popular pe leaderboard.


In [ ]:

if HAS_XGB:
    xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        reg_alpha=0.0,
        reg_lambda=1.0
    )
    
    pipe_xgb = Pipeline(steps=[("preprocess", preprocess),
                              ("model", xgb)])
    
    rmse_scores_xgb = rmse_cv(pipe_xgb, X, y)
    print("XGB CV RMSE (log-target):", rmse_scores_xgb.mean().round(4), "±", rmse_scores_xgb.std().round(4))
    
    pipe_xgb.fit(X, y)
    preds_log_xgb = pipe_xgb.predict(X_test)
    preds_xgb = np.expm1(preds_log_xgb)
    
    submission_xgb = pd.DataFrame({ID_COL: test[ID_COL], TARGET: preds_xgb})
    submission_path_xgb = "submission_house_prices_xgb.csv"
    submission_xgb.to_csv(submission_path_xgb, index=False)
    print(f"Am salvat: {submission_path_xgb}")
    display(submission_xgb.head())
else:
    print("XGBoost nu este disponibil în acest runtime; sari peste această secțiune sau instalează pachetul.")
